# 0. Environment Setup

## Library Import

Loads all libraries required for the full pipeline.

| Library | Purpose |
|---|---|
| `pandas` | DataFrame operations |
| `datetime` | Date math for Monday edge-case routing |
| `requests` | HTTP calls to iQor SFTP server and Qualtrics API |
| `json` | Parse Qualtrics API responses |
| `urllib3` | SSL warning suppression |
| `difflib.get_close_matches` | Fuzzy-match df column names to Qualtrics `questionName` fields |
| `ctypes`, `threading`, `time` | Windows popup notifications (non-blocking) |
| `os` | Build local file paths |
| `urllib.parse.quote` | URL-encode filenames with spaces for SFTP HTTP requests |
| `io.StringIO` | Read CSV text from HTTP response as a file-like object |

In [2]:
import pandas as pd
from datetime import date, timedelta
import re
import requests
import json
import urllib3
from difflib import get_close_matches
import ctypes
import threading
import time
import os
from urllib.parse import quote
requests.packages.urllib3.disable_warnings()
from io import StringIO

## Qualtrics Credentials

Sets the API token, data center, and target survey ID for each OpCo.

| Variable | Description |
|---|---|
| `API_TOKEN` | Qualtrics API authentication token |
| `DATA_CENTER` | Qualtrics data center identifier (e.g. `iad1`) |
| `SURVEYS_ID` | Dict mapping `"CMP"`, `"RGE"`, `"NSE"` to their Qualtrics survey IDs |

> âš ï¸ **Security:** Move credentials to environment variables or a `.env` file before sharing or deploying.

In [3]:
# ============================================================
# CONFIGURATION 
# ============================================================

API_TOKEN = "dZZueEgbaPrCuSTXEtIp2sz0EqlJNxg93jYEod7U"   # <-- insert your token
DATA_CENTER = "iad1"
SURVEYS_ID = {
    "CMP" : "SV_6zfVwcNb8KrpiPI",
    "RGE" : "SV_4HFHCtt06kaQv0a",
    "NSE" : "SV_eyAxdOqDXvF3qu2"
}

## iQor SFTP Credentials

Sets the base URL, username, and password for iQor's managed file transfer server.

| Variable | Description |
|---|---|
| `BASE_URL` | iQor MFT server base URL |
| `USERNAME` | SFTP account username |
| `PASSWORD` | SFTP account password |

> âš ï¸ **Security:** Move credentials to environment variables before sharing or deploying.

In [4]:
"""
BASE_URL = 'https://mft.iqor.com'
USERNAME = 'Iberdrola'
PASSWORD = 'KaC#9eta'
"""

"\nBASE_URL = 'https://mft.iqor.com'\nUSERNAME = 'Iberdrola'\nPASSWORD = 'KaC#9eta'\n"

In [5]:
BASE_URL = 'https://mft.iqor.com'
USERNAME = 'Avangrid_Automation'
PASSWORD = 'K3P#9e$td'

## Popup Notifications

Defines two helper functions for Windows message box dialogs.

- `popup_info(message)` â€” success dialog (blue icon)
- `popup_error(message)` â€” error/warning dialog (red icon)

Both run the dialog in a background thread (non-blocking) and auto-close after `timeout` seconds.

> âš ï¸ **Windows only.** Replace with email or Teams notification when deploying to Azure or Linux.

In [6]:
def popup(message, title="Info", timeout=10, is_error=False):
    MB_OK = 0x0
    icon = 0x10 if is_error else 0x40  # ERROR vs INFORMATION
    
    def show_box():
        ctypes.windll.user32.MessageBoxW(0, message, title, MB_OK | icon)
    
    t = threading.Thread(target=show_box)
    t.start()
    t.join(timeout=timeout)
    
    hwnd = ctypes.windll.user32.FindWindowW(None, title)
    if hwnd:
        ctypes.windll.user32.PostMessageW(hwnd, 0x0010, 0, 0)

# Keep your original names as simple wrappers if you want
def popup_info(message, title="Success", timeout=10):
    popup(message, title, timeout, is_error=False)

def popup_error(message, title="Error", timeout=10):
    popup(message, title, timeout, is_error=True)

## SharePoint / OneDrive Output Folder

Resolves the local OneDrive sync path for each OpCo to save output files.

- `get_sharepoint_folder(opco)` â†’ full folder path string
- `get_output_path(filename, opco)` â†’ `(full_path, folder_exists_bool)`

Falls back gracefully â€” if the SharePoint folder is not synced locally, the file is written to the current working directory instead.

| OpCo | Subfolder |
|---|---|
| CMP | `iQor_CMP` |
| RGE | `iQor_RGE` |
| NSE | `iQor_NYSEG` |

In [7]:
def get_onedrive_path():
    return os.path.join(os.path.expanduser("~"), "OneDrive - IBERDROLA S.A")

def get_sharepoint_folder(opco):
    """
    Returns the landing folder path for a given OpCo.
    opco: 'CMP', 'RGE', or 'NSE'
    """
    OPCO_FOLDERS = {
        "CMP": "iQor_CMP",
        "RGE": "iQor_RGE",
        "NSE": "iQor_NYSEG"
    }
    
    if opco not in OPCO_FOLDERS:
        raise ValueError(f"Unknown OpCo: {opco}. Must be one of {list(OPCO_FOLDERS.keys())}")
    
    onedrive_root = get_onedrive_path()
    return os.path.join(
        onedrive_root,
        "iQor-Avangrid - General",
        OPCO_FOLDERS[opco]
    )

def get_output_path(filename, opco):
    """
    Returns (full_path, folder_exists) for a given filename and OpCo.
    """
    sharepoint_folder = get_sharepoint_folder(opco)
    if os.path.exists(sharepoint_folder):
        return os.path.join(sharepoint_folder, filename), True
    return filename, False

# 1. Extract

## Extract â€” Pull Latest File from iQor SFTP

**What it does:**
1. Opens an authenticated HTTP session to iQor's MFT server
2. Lists each OpCo's survey report folder on the server
3. Identifies the most recent `Daily Survey Report_YYYYMMDD.csv` (skips `Triage` files)
4. Downloads the file and parses it into a DataFrame

**Input:** iQor SFTP credentials (`BASE_URL`, `USERNAME`, `PASSWORD`) + `SFTP_FOLDERS` dict

**Output:**
- `dataframes = {"CMP": df, "RGE": df, "NSE": df}` â€” one raw DataFrame per OpCo
- `raw_files = {"CMP": (filename, raw_text), ...}` â€” raw CSV text for local audit copies
- `df_cmp`, `df_rge`, `df_nse` â€” convenience shortcuts

> The `time.sleep(2)` warmup calls are intentional â€” the iQor server requires a brief pause after the initial handshake before directory listings return correct results.

In [8]:
"""session = requests.Session()
session.verify = False
session.auth = (USERNAME, PASSWORD)
session.get(BASE_URL + '/files', timeout=15)"""

"session = requests.Session()\nsession.verify = False\nsession.auth = (USERNAME, PASSWORD)\nsession.get(BASE_URL + '/files', timeout=15)"

In [9]:
# ============================================================
# EXTRACT
# ============================================================

session = requests.Session()
session.verify = False
session.auth = (USERNAME, PASSWORD)

# Warmup - allow server to fully establish session
session.get(BASE_URL + '/files', timeout=15)
time.sleep(2)
session.get(BASE_URL + '/Report/Survey%20Reports/', timeout=15)
time.sleep(2)

SFTP_FOLDERS = {
    "CMP": "/Report/Survey%20Reports/CMP/",
    "RGE": "/Report/Survey%20Reports/RGE/",
    "NSE": "/Report/Survey%20Reports/NSE/",
}

def list_dir(path):
    r = session.get(BASE_URL + path, timeout=15)
    files, folders = [], []
    for line in r.text.strip().splitlines()[1:]:
        parts = line.split()
        if len(parts) < 9:
            continue
        perms, _, owner, group, size, month, day, time_str, *name_parts = parts
        name = ' '.join(name_parts)
        if name in ('.', '..'):
            continue
        is_dir = perms.startswith('d')
        entry = {
            'name' : name,
            'size' : int(size),
            'date' : f'{month} {day} {time_str}',
            'type' : 'DIR' if is_dir else 'FILE'
        }
        (folders if is_dir else files).append(entry)
    return files, folders

def get_latest_file(opco):
    folder_path  = SFTP_FOLDERS[opco]
    files, _     = list_dir(folder_path)
    time.sleep(1)
    reports      = [
        f for f in files
        if 'Daily Survey Report_' in f['name']
        and 'Triage' not in f['name']
    ]
    if not reports:
        raise FileNotFoundError(f"No Daily Survey Reports found for {opco}")
    reports.sort(key=lambda f: f['name'].split('_')[-1].replace('.csv', ''))
    latest       = reports[-1]
    print(f"[{opco}] Latest file : {latest['name']}")
    print(f"[{opco}] Server date : {latest['date']}")
    encoded_name = quote(latest['name'])
    r            = session.get(BASE_URL + folder_path + encoded_name, timeout=30)
    r.raise_for_status()
    parts        = latest['name'].replace('.csv', '').split('_')
    company      = parts[0].split()[0]
    # âš ï¸ TYPE THESE TWO LINES BY HAND IN JUPYTER
    report_date  = pd.to_datetime(parts[-1], format='%Y%m%d')
    df           = pd.read_csv(StringIO(r.text))
    # âš ï¸ END
    #df['company']     = company
    #df['report_date'] = report_date
    print(f"[{opco}] Shape       : {df.shape}")
    return df, latest['name'], r.text

# --- Run ---
dataframes = {}
raw_files  = {}
for opco in SFTP_FOLDERS:
    try:
        df, filename, raw_text = get_latest_file(opco)
        dataframes[opco]       = df
        raw_files[opco]        = (filename, raw_text)
    except Exception as e:
        print(f"[{opco}] ERROR: {e}")

df_cmp = dataframes.get("CMP")
df_rge = dataframes.get("RGE")
df_nse = dataframes.get("NSE")

[CMP] Latest file : CMP Daily Survey Report_20260629.csv
[CMP] Server date : Jun 30 07:01:21
[CMP] Shape       : (66, 15)
[RGE] Latest file : RGE Daily Survey Report_20260629.csv
[RGE] Server date : Jun 30 07:01:16
[RGE] Shape       : (198, 14)
[NSE] Latest file : NSE Daily Survey Report_20260629.csv
[NSE] Server date : Jun 30 07:01:10
[NSE] Shape       : (391, 14)


In [10]:
# ============================================================
# LOCAL ONLY â€” Save raw CSVs to SharePoint for audit/pivot use
# Remove this entire cell when deploying to Azure
# ============================================================

for opco, (filename, raw_text) in raw_files.items():
    try:
        local_path, folder_exists = get_output_path(filename, opco)
        if folder_exists:
            with open(local_path, 'w', encoding='utf-8', newline='') as f:
                f.write(raw_text)
            print(f"[{opco}] âœ“ Saved to : {local_path}")
        else:
            print(f"[{opco}] âš  Skipped  : Folder not found â€” {local_path}")
    except PermissionError:
        print(f"[{opco}] âš  Skipped  : File is open in Excel or locked â€” {filename}")
    except Exception as e:
        print(f"[{opco}] âš  Skipped  : {e}")

print("\nLocal save complete â€” continuing pipeline...")

[CMP] âœ“ Saved to : C:\Users\E978423\OneDrive - IBERDROLA S.A\iQor-Avangrid - General\iQor_CMP\CMP Daily Survey Report_20260629.csv
[RGE] âœ“ Saved to : C:\Users\E978423\OneDrive - IBERDROLA S.A\iQor-Avangrid - General\iQor_RGE\RGE Daily Survey Report_20260629.csv
[NSE] âœ“ Saved to : C:\Users\E978423\OneDrive - IBERDROLA S.A\iQor-Avangrid - General\iQor_NYSEG\NSE Daily Survey Report_20260629.csv

Local save complete â€” continuing pipeline...


In [11]:
display(df_cmp.head(3))
display(df_nse.head(3))
display(df_rge.head(3))
print("hello world")

,ID,Name,Date_Time,Work_Group,InteractionID,Phone_Number,Survey_Name,CSAT1,NPS,I_C,C_K,FCR,Call_Reason,Survey_Status_Count,Survey_Status
0,34795476,jasmine.robinson9,06/29/2026 07:38:20,CMP.USUT.CS.RESCRCL,710269613726,anonymous,CMP IQR Survey w/ NPS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ABANDONED
1,38999525,cachae.perry,06/29/2026 07:39:49,CMP.USUT.CS.RESCRCL,710269614371,2076605564,CMP IQR Survey w/ NPS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ABANDONED
2,34795424,tanique.russell,06/29/2026 07:41:06,CMP.USUT.CS.RESCRCL,710269614405,2078979445,CMP IQR Survey w/ NPS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ABANDONED


,ID,Name,Date,WorkGroup,InteractionID,Telephone,SurveyName,NPS,FCR,CSAT,E_H,C_E,CallReason,SurveyStatus
0,38870048,kendrell.lewis1,06/29/2026 07:10:41,NSE.USUT.FE.NYCRCL,710269607130,7163923224,NSE IQR Survey w/ NPS,NaN,NaN,NaN,NaN,NaN,NaN,ABANDONED
1,39062909,jacqueline.gabriel,06/29/2026 07:12:03,NSE.USUT.FE.GEN,710269606270,6072262125,NSE IQR Survey w/ NPS,NaN,NaN,NaN,NaN,NaN,NaN,ABANDONED
2,60920556,alexandria.colvin,06/29/2026 07:13:00,NSE.USUT.FE.GEN,710269605646,7164335011,NSE IQR Survey w/ NPS,10.0,1.0,5.0,1.0,5.0,3.0,COMPLETED


,ID,Name,Date,WorkGroup,InteractionID,Telephone,SurveyName,NPS,FCR,CSAT,E_H,C_E,CallReason,SurveyStatus
0,43978728,shuhun.baloch,06/29/2026 07:20:22,RGE.USUT.FE.RGCRCL,710269605057,5854450960,RGE IQR Survey w/ NPS,10.0,1.0,5.0,4.0,4.0,3.0,COMPLETED
1,44081434,kayla.hunt1,06/29/2026 07:20:46,RGE.USUT.FE.RGCRCL,710269604699,5856988135,RGE IQR Survey w/ NPS,10.0,1.0,5.0,5.0,5.0,3.0,COMPLETED
2,69022815,matheresita.inciong,06/29/2026 07:25:43,RGE.USUT.FE.RGCRCL,710269608000,5857647942,RGE IQR Survey w/ NPS,0.0,0.0,4.0,3.0,5.0,1.0,COMPLETED


hello world


In [12]:
print("CMP:",df_cmp.columns.tolist())
print("NSE:",df_nse.columns.tolist())
print("RGE:",df_rge.columns.tolist())

CMP: ['ID', 'Name', 'Date_Time', 'Work_Group', 'InteractionID', 'Phone_Number', 'Survey_Name', 'CSAT1', 'NPS', 'I_C', 'C_K', 'FCR', 'Call_Reason', 'Survey_Status_Count', 'Survey_Status']
NSE: ['ID', 'Name', 'Date', 'WorkGroup', 'InteractionID', 'Telephone', 'SurveyName', 'NPS', 'FCR', 'CSAT', 'E_H', 'C_E', 'CallReason', 'SurveyStatus']
RGE: ['ID', 'Name', 'Date', 'WorkGroup', 'InteractionID', 'Telephone', 'SurveyName', 'NPS', 'FCR', 'CSAT', 'E_H', 'C_E', 'CallReason', 'SurveyStatus']


# 2. Transform

## Transform â€” Rename Columns & Standardize Types

**What it does:**
1. Renames source columns to consistent names â€” CMP uses different field names than RGE/NSE
2. Standardizes `Date/Time` to `MM/DD/YYYY HH:MM:SS`
3. Casts score columns to nullable integers (`Int64`) â€” preserves `NaN` for missing responses

**Input:** Raw DataFrame from extract + `opco` string (`"CMP"`, `"RGE"`, or `"NSE"`)

**Output:** Transformed DataFrame with consistent column names and types

| OpCo | Renames applied |
|---|---|
| CMP | `Date_Time`â†’`Date/Time`, `CSAT1`â†’`CSAT`, `Survey_Status`â†’`Survey Completion` |
| RGE | `Date`â†’`Date/Time`, `Telephone`â†’`Phone Number` |
| NSE | `Date`â†’`Date/Time`, `Telephone`â†’`Phone Number` |

In [13]:
# ============================================================
# TRANSFORM
# ============================================================

# --- Column rename maps per OpCo ---
RENAME_MAP = {
    "CMP": {
        "Date_Time": "Date/Time",
        "CSAT1"    : "CSAT",
        "Survey_Status" : "Survey Completion",
    },
    "RGE": {
        "Date"     : "Date/Time",
        "Telephone" : "Phone Number",
    },
    "NSE": {
        "Date"     : "Date/Time",
        "Telephone" : "Phone Number",
    },
}

# --- Score columns to cast to integer per OpCo ---
SCORE_COLS = {
    "CMP": ["NPS", "CSAT", "Call_Reason", "Survey_Status_Count"],
    "RGE": ["NPS", "FCR", "CSAT", "E_H", "C_E", "CallReason"],
    "NSE": ["NPS", "FCR", "CSAT", "E_H", "C_E", "CallReason"],
}

def transform(df, opco):
    df = df.copy()

    # Step 1 â€” Rename columns
    df = df.rename(columns=RENAME_MAP.get(opco, {}))

    # Step 2 â€” Fix Date/Time format
    if "Date/Time" in df.columns:
        df["Date/Time"] = pd.to_datetime(df["Date/Time"], errors="coerce")
        df["Date/Time"] = df["Date/Time"].dt.strftime("%m/%d/%Y %H:%M:%S")

    # Step 3 â€” Cast score columns to integer
    for col in SCORE_COLS[opco]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    print(f"[{opco}] Transform complete | Shape: {df.shape}")
    return df

# --- Run ---
df_cmp = transform(df_cmp, "CMP")
df_rge = transform(df_rge, "RGE")
df_nse = transform(df_nse, "NSE")

[CMP] Transform complete | Shape: (66, 15)
[RGE] Transform complete | Shape: (198, 14)
[NSE] Transform complete | Shape: (391, 14)


In [14]:
# DEBUG - run after transform, before prepare
print("C_K in df_cmp.columns:", "C_K" in df_cmp.columns)
print("C_K all null:", df_cmp["C_K"].isna().all())
print("C_K dtype:", df_cmp["C_K"].dtype)
print("C_K sample:\n", df_cmp["C_K"].value_counts(dropna=False))

C_K in df_cmp.columns: True
C_K all null: False
C_K dtype: object
C_K sample:
 C_K
NaN    43
#      21
0       2
Name: count, dtype: int64


## Prepare â€” Step 1: Validate & Tag

**What it does:**
1. Warns via popup if any unexpected columns are detected in the DataFrame
2. Raises `ValueError` if required score columns are missing
3. Drops fully-empty columns
4. Forward-fills `ID` and `Name` (handles repeated rows for the same agent)
5. Tags rows where `Work Group` contains `"test"` (case-insensitive) as `Tag = "Test"`, others as `""`

**Input:** Transformed DataFrame + `opco`

**Output:** Validated DataFrame with `Tag` column added

In [15]:
# ============================================================
# PREPARE FOR QUALTRICS â€” Step 1: Validate & Tag
# ============================================================

EXPECTED_COLS = {
    "CMP": ["ID", "Name", "Date/Time", "Work_Group", "InteractionID", "Phone_Number",
            "Survey_Name", "CSAT", "NPS", "I_C", "C_K", "FCR", "Call_Reason",
            "Survey_Status_Count", "Survey Completion", "Tag"],
    "RGE": ["ID", "Name", "Date/Time", "WorkGroup", "InteractionID", "Phone Number",
            "SurveyName", "NPS", "FCR", "CSAT", "E_H", "C_E", "CallReason",
            "SurveyStatus", "Tag"],
    "NSE": ["ID", "Name", "Date/Time", "WorkGroup", "InteractionID", "Phone Number",
            "SurveyName", "NPS", "FCR", "CSAT", "E_H", "C_E", "CallReason",
            "SurveyStatus", "Tag"],
}

REQUIRED_SCORE_COLS = {
    "CMP": ["CSAT", "NPS", "I_C", "C_K", "FCR", "Call_Reason"],
    "RGE": ["NPS", "FCR", "CSAT", "E_H", "C_E", "CallReason"],
    "NSE": ["NPS", "FCR", "CSAT", "E_H", "C_E", "CallReason"],
}

WORKGROUP_COL = {
    "CMP": "Work_Group",
    "RGE": "WorkGroup",
    "NSE": "WorkGroup",
}

def validate_and_tag(df, opco):
    df = df.copy()

    # Step 1 â€” Warn on unknown columns
    unknown_cols = [c for c in df.columns if c not in EXPECTED_COLS[opco]]
    if unknown_cols:
        popup_error(
            f"[{opco}] Unknown columns detected:\n{unknown_cols}",
            title="ETL Warning"
        )
        print(f"[{opco}] âš  Unknown columns: {unknown_cols}")

    # Step 2 â€” Validate required scoring columns
    required = REQUIRED_SCORE_COLS[opco]
    missing  = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"[{opco}] Missing required scoring columns: {missing}")

    # Step 3 â€” Drop empty columns
    df = df.dropna(axis=1, how="all")

    # Step 4 â€” Forward fill ID and Name
    df[["ID", "Name"]] = df[["ID", "Name"]].ffill()

    # Step 5 â€” Tag test rows (case insensitive)
    wg_col    = WORKGROUP_COL[opco]
    df["Tag"] = df[wg_col].str.contains("test", case=False, na=False).map({
        True : "Test",
        False: ""
    })

    print(f"[{opco}] Validate & tag complete | Shape: {df.shape}")
    if df["Tag"].eq("Test").any():
        print(f"[{opco}] âš  Test rows found: {df['Tag'].eq('Test').sum()}")

    return df

# --- Run ---
df_cmp = validate_and_tag(df_cmp, "CMP")
df_rge = validate_and_tag(df_rge, "RGE")
df_nse = validate_and_tag(df_nse, "NSE")

[CMP] Validate & tag complete | Shape: (66, 16)
[RGE] Validate & tag complete | Shape: (198, 15)
[NSE] Validate & tag complete | Shape: (391, 15)


## Filter â€” Remove Incomplete Responses

**What it does:** Keeps only rows where **all** score columns are filled in. Any row with at least one missing score â€” or with a `COMPLETED`/`ABANDONED` status â€” is excluded before upload to Qualtrics.

**Score columns checked per OpCo:**

| OpCo | Columns |
|---|---|
| CMP | `CSAT`, `NPS`, `I_C`, `C_K`, `FCR`, `Call_Reason` |
| RGE | `NPS`, `FCR`, `CSAT`, `E_H`, `C_E`, `CallReason` |
| NSE | `NPS`, `FCR`, `CSAT`, `E_H`, `C_E`, `CallReason` |

**Input:** Tagged DataFrame + `opco`

**Output:** Filtered DataFrame (completed responses only) + printed summary:
```
[CMP] Original records  : 83
[CMP] Completed records : 38
[CMP] Excluded          : 45
```

In [16]:
# ============================================================
# FILTER â€” Remove Incomplete Responses
# ============================================================

COMPLETION_COLS = {
    "CMP": ["CSAT", "NPS", "I_C", "C_K", "FCR", "Call_Reason"],
    "RGE": ["NPS", "FCR", "CSAT", "E_H", "C_E", "CallReason"],
    "NSE": ["NPS", "FCR", "CSAT", "E_H", "C_E", "CallReason"],
}

STATUS_COL = {
    "CMP": "Survey Completion",
    "RGE": "SurveyStatus",
    "NSE": "SurveyStatus",
}

def filter_completed(df, opco):
    total      = len(df)
    check_cols = [c for c in COMPLETION_COLS[opco] if c in df.columns]

    # A row is complete only if ALL score columns are non-null
    mask = df[check_cols].notna().all(axis=1)

    # Belt-and-suspenders: also drop any remaining ABANDONED status rows
    status_col = STATUS_COL.get(opco)
    if status_col and status_col in df.columns:
        mask = mask & (df[status_col].str.upper() != "ABANDONED")

    completed = df[mask].reset_index(drop=True)
    excluded  = total - len(completed)

    print(f"[{opco}] Original records  : {total}")
    print(f"[{opco}] Completed records : {len(completed)}")
    print(f"[{opco}] Excluded          : {excluded}")
    return completed

# --- Run ---
df_cmp = filter_completed(df_cmp, "CMP")
df_rge = filter_completed(df_rge, "RGE")
df_nse = filter_completed(df_nse, "NSE")

[CMP] Original records  : 66
[CMP] Completed records : 18
[CMP] Excluded          : 48


[RGE] Original records  : 198
[RGE] Completed records : 110
[RGE] Excluded          : 88
[NSE] Original records  : 391
[NSE] Completed records : 226
[NSE] Excluded          : 165


## Prepare â€” Step 2: Qualtrics API Field Mapping

**What it does:**
1. Calls the Qualtrics Survey API to retrieve official question names (`questionName`), question text (`questionText`), and QIDs for the target survey
2. Fuzzy-matches each DataFrame column name to Qualtrics `questionName` values (similarity cutoff: 0.6)
3. Renames DataFrame columns to their official Qualtrics names
4. Builds the **3-row header block** required by the Qualtrics Import Responses API:
   - Row 1: `questionName` â€” becomes the CSV column headers
   - Row 2: `questionText` â€” human-readable question labels
   - Row 3: `{"ImportId": "QIDx_TEXT"}` â€” Qualtrics internal field mapping
5. Stacks headers + data into the final upload-ready DataFrame

**Input:** Filtered DataFrame from `filter_completed()` + `opco`

**Output:** Upload-ready DataFrame â€” shape `(completed_rows + 2, columns)`

In [17]:
def prepare_for_qualtrics(df, opco):
    survey_id = SURVEYS_ID[opco]
    
    url     = f"https://{DATA_CENTER}.qualtrics.com/API/v3/surveys/{survey_id}"
    headers = {"X-API-TOKEN": API_TOKEN}
    
    response = requests.get(url, headers=headers, verify=False)
    response.raise_for_status()
    
    survey_json   = response.json()
    label_map     = survey_json["result"]["questions"]
    
    qid_to_name   = {qid: q["questionName"] for qid, q in label_map.items()}
    qid_to_text   = {qid: q["questionText"] for qid, q in label_map.items()}
    name_to_qid   = {name: qid for qid, name in qid_to_name.items()}
    qualtrics_names = list(name_to_qid.keys())

    # Step 1 â€” Fuzzy match df columns to Qualtrics questionName
    mapped_cols = {}
    used_names  = set()
    for col in df.columns:
        match = get_close_matches(col, qualtrics_names, n=1, cutoff=0.6)
        if match:
            new_name = match[0]
            if new_name in used_names:
                new_name = col
            mapped_cols[col] = new_name
            used_names.add(new_name)
        else:
            mapped_cols[col] = col

    df = df.rename(columns=mapped_cols)

    # Step 2 â€” Build questionText row
    row2 = []
    for col in df.columns:
        qid = name_to_qid.get(col)
        row2.append(qid_to_text.get(qid, "") if qid else "")

    # Step 3 â€” Build ImportId row
    row3 = []
    for col in df.columns:
        qid = name_to_qid.get(col)
        row3.append(f'{{"ImportId": "{qid}_TEXT"}}' if qid else "")

    # Step 4 â€” Stack headers + data
    hdr1     = pd.DataFrame([list(df.columns)], columns=df.columns)
    hdr2     = pd.DataFrame([row2],             columns=df.columns)
    hdr3     = pd.DataFrame([row3],             columns=df.columns)
    df_final = pd.concat([hdr1, hdr2, hdr3, df.reset_index(drop=True)], ignore_index=True)

    # Step 5 â€” Remove duplicate header row
    df_final = df_final.iloc[1:].reset_index(drop=True)

    print(f"[{opco}] Qualtrics prep complete | Shape: {df_final.shape}")
    display(df_final.head(5))
    
    return df_final

# --- Run ---
df_cmp_q = prepare_for_qualtrics(df_cmp, "CMP")
df_rge_q = prepare_for_qualtrics(df_rge, "RGE")
df_nse_q = prepare_for_qualtrics(df_nse, "NSE")

[CMP] Qualtrics prep complete | Shape: (20, 16)


,ID,Name,Date/Time,Work Group,Interaction ID,Phone Number,Survey Name,CSAT,NPS,I_C,C_K,FCR,Call_Reason,Survey Status Count,Survey Completion,Tag
0,ID,Name,Date/Time,Work Group,Interaction ID,Phone Number,Survey Name,CSAT,NPS,I_C,C_K,FCR,Call_Reason,Survey Status Count,Survey Completion,Tag
1,"{""ImportId"": ""QID1_TEXT""}","{""ImportId"": ""QID2_TEXT""}","{""ImportId"": ""QID3_TEXT""}","{""ImportId"": ""QID4_TEXT""}","{""ImportId"": ""QID5_TEXT""}","{""ImportId"": ""QID6_TEXT""}","{""ImportId"": ""QID7_TEXT""}","{""ImportId"": ""QID9_TEXT""}","{""ImportId"": ""QID8_TEXT""}","{""ImportId"": ""QID10_TEXT""}","{""ImportId"": ""QID11_TEXT""}","{""ImportId"": ""QID12_TEXT""}","{""ImportId"": ""QID13_TEXT""}","{""ImportId"": ""QID15_TEXT""}","{""ImportId"": ""QID16_TEXT""}","{""ImportId"": ""QID14_TEXT""}"
2,56441398,sean.case,06/29/2026 07:42:01,CMP.USUT.CS.RESCRCL,710269613439,2627252384,CMP IQR Survey w/ NPS,5,1,#,#,#,1,6,COMPLETED,
3,34795386,evonta.garrett,06/29/2026 08:46:34,CMP.USUT.CS.RESCRCL,710269644761,2077981740,CMP IQR Survey w/ NPS,5,4,#,#,#,3,6,COMPLETED,
4,39761737,brianna.gay,06/29/2026 08:55:46,CMP.USUT.CS.RESCRCL,710269657735,2073515481,CMP IQR Survey w/ NPS,5,3,#,#,#,1,6,COMPLETED,


[RGE] Qualtrics prep complete | Shape: (112, 15)


,ID,Name,Date/Time,Work Group,Interaction ID,Phone Number,Survey Name,NPS,FCR,CSAT,E_H,C_E,Call Reason,Survey Status,Tag
0,ID,Name,Date/Time,Work Group,Interaction ID,Phone Number,Survey Name,NPS,FCR,CSAT,Ease of Help,Clear Explanation,Call Reason,Survey Status,Tag
1,"{""ImportId"": ""QID1_TEXT""}","{""ImportId"": ""QID2_TEXT""}","{""ImportId"": ""QID3_TEXT""}","{""ImportId"": ""QID4_TEXT""}","{""ImportId"": ""QID5_TEXT""}","{""ImportId"": ""QID6_TEXT""}","{""ImportId"": ""QID7_TEXT""}","{""ImportId"": ""QID9_TEXT""}","{""ImportId"": ""QID12_TEXT""}","{""ImportId"": ""QID8_TEXT""}","{""ImportId"": ""QID10_TEXT""}","{""ImportId"": ""QID11_TEXT""}","{""ImportId"": ""QID13_TEXT""}","{""ImportId"": ""QID14_TEXT""}","{""ImportId"": ""QID15_TEXT""}"
2,43978728,shuhun.baloch,06/29/2026 07:20:22,RGE.USUT.FE.RGCRCL,710269605057,5854450960,RGE IQR Survey w/ NPS,10,1,5,4,4,3,COMPLETED,
3,44081434,kayla.hunt1,06/29/2026 07:20:46,RGE.USUT.FE.RGCRCL,710269604699,5856988135,RGE IQR Survey w/ NPS,10,1,5,5,5,3,COMPLETED,
4,69022815,matheresita.inciong,06/29/2026 07:25:43,RGE.USUT.FE.RGCRCL,710269608000,5857647942,RGE IQR Survey w/ NPS,0,0,4,3,5,1,COMPLETED,


[NSE] Qualtrics prep complete | Shape: (228, 15)


,ID,Name,Date/Time,Work Group,Interaction ID,Phone Number,Survey Name,NPS,FCR,CSAT,E_H,C_E,Call Reason,Survey Status,Tag
0,ID,Name,Date/Time,Work Group,Interaction ID,Phone Number,Survey Name,NPS,FCR,CSAT,Ease of Help,Clear Explanation,Call Reason,Survey Status,Tag
1,"{""ImportId"": ""QID1_TEXT""}","{""ImportId"": ""QID2_TEXT""}","{""ImportId"": ""QID3_TEXT""}","{""ImportId"": ""QID4_TEXT""}","{""ImportId"": ""QID5_TEXT""}","{""ImportId"": ""QID6_TEXT""}","{""ImportId"": ""QID7_TEXT""}","{""ImportId"": ""QID9_TEXT""}","{""ImportId"": ""QID12_TEXT""}","{""ImportId"": ""QID8_TEXT""}","{""ImportId"": ""QID10_TEXT""}","{""ImportId"": ""QID11_TEXT""}","{""ImportId"": ""QID13_TEXT""}","{""ImportId"": ""QID14_TEXT""}","{""ImportId"": ""QID15_TEXT""}"
2,60920556,alexandria.colvin,06/29/2026 07:13:00,NSE.USUT.FE.GEN,710269605646,7164335011,NSE IQR Survey w/ NPS,10,1,5,1,5,3,COMPLETED,
3,69353654,symba.sebastian,06/29/2026 07:25:34,NSE.USUT.FE.GEN,710269607364,5855573534,NSE IQR Survey w/ NPS,10,1,5,5,5,1,COMPLETED,
4,38869988,sherry.stdennis,06/29/2026 07:26:24,NSE.USUT.FE.NYCRCL,710269609063,8454282579,NSE IQR Survey w/ NPS,10,1,5,5,5,3,COMPLETED,


# 3. Load

## Load â€” Save DataFrame to CSV

**What it does:** Writes any DataFrame to a CSV file at the specified path.

**Input:**
- `df` â€” any DataFrame (raw, cleaned, or Qualtrics-ready)
- `output_path` â€” full file path including filename

**Output:** UTF-8 CSV file written to disk (no index column)

Two files are saved per OpCo per run:
1. **Cleaned file** â†’ SharePoint folder (for audit and manual pivot use)
2. **Qualtrics temp file** â†’ local working directory (uploaded to Qualtrics, then can be deleted)

In [18]:
def load_data(df, output_path):
    df.to_csv(output_path, index=False, encoding="utf-8")

    return


## Date Logic â€” Source Folder Routing

**What it does:** Computes the effective date used for building the source file path and output filename. Handles the Monday edge case â€” since no reports are generated on Sunday, a Monday run looks for Saturday's file.

**Input:** System date (`date.today()`)

**Output:** Date variables used downstream

| Variable | Example | Used for |
|---|---|---|
| `source_year` | `2026` | Year component of file path |
| `source_month` | `"June"` | Month folder name |
| `source_day` | `"15"` | Zero-padded day folder name |
| `landing_yesterday` | `2026-06-14` | Output filename date suffix |

In [19]:
# Extract Date fields
today = date.today()

# If today is Monday (weekday() == 0), use last Saturday
if today.weekday() == 0:
    effective_date = today - timedelta(days=2)
else:
    effective_date = today

# Extract Date fields
source_year = effective_date.year
source_month = effective_date.strftime("%B")
source_day = effective_date.strftime("%d")
source_weekday = effective_date.weekday()
landing_yesterday = effective_date - timedelta(days=1)

#print(source_year, source_month, source_day, source_weekday, landing_yesterday)


## Upload to Qualtrics

**What it does:** POSTs the Qualtrics-ready CSV to the Import Responses API, then polls the job status until the upload completes or fails.

**Input:**
- `file_path` â€” path to the Qualtrics-formatted temp CSV
- `DATA_CENTER`, `SURVEY_ID`, `API_TOKEN` â€” Qualtrics credentials

**Output:** Final job status dict from the Qualtrics API

**API flow:**
1. `POST /API/v3/surveys/{SURVEY_ID}/import-responses` â†’ get `progressId`
2. Poll `GET /API/v3/surveys/{SURVEY_ID}/import-responses/{progressId}` every 3 s until `status == "complete"` or `"failed"`

> Without polling, a `200 - OK` response only means the job was **accepted**, not finished. This function waits for actual completion before returning.

In [20]:
def upload_to_qualtrics(file_path, DATA_CENTER, SURVEY_ID, API_TOKEN):
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    url     = f"https://{DATA_CENTER}.qualtrics.com/API/v3/surveys/{SURVEY_ID}/import-responses"
    headers = {"X-API-TOKEN": API_TOKEN, "Content-Type": "text/csv", "charset": "UTF-8"}

    print(f"\nUploading: {file_path}")
    with open(file_path, "rb") as f:
        resp = requests.post(url, headers=headers, data=f, verify=False)
    resp.raise_for_status()

    result      = resp.json()
    progress_id = result["result"]["progressId"]
    print(f"Job accepted | progressId: {progress_id}")

    # Poll until the import job finishes
    poll_url     = f"https://{DATA_CENTER}.qualtrics.com/API/v3/surveys/{SURVEY_ID}/import-responses/{progress_id}"
    poll_headers = {"X-API-TOKEN": API_TOKEN}

    for attempt in range(30):
        time.sleep(3)
        poll_resp   = requests.get(poll_url, headers=poll_headers, verify=False)
        poll_result = poll_resp.json()["result"]
        pct         = poll_result.get("percentComplete", 0)
        status      = poll_result.get("status", "unknown")
        print(f"  [{attempt + 1:02d}] {status} | {pct:.0f}%")

        if status == "complete":
            print("Upload confirmed complete.")
            return poll_result
        if status == "failed":
            raise RuntimeError(f"Qualtrics import failed: {poll_result}")

    raise TimeoutError("Upload did not complete within 90 seconds.")

# Execute

Runs the full ETL pipeline end-to-end for all three OpCos (CMP, RGE, NSE).

**Pipeline per OpCo:**
```
1. transform(df, opco)             â†’ rename columns, standardize types
2. validate_and_tag(df, opco)      â†’ validate columns, forward-fill ID/Name, tag test rows
3. filter_completed(df, opco)      â†’ keep only fully-answered responses
4. load_data(df, sharepoint_path)  â†’ save cleaned file to SharePoint
5. prepare_for_qualtrics(df, opco) â†’ map to Qualtrics field names + build 3-row header
6. load_data(df_q, temp_path)      â†’ save Qualtrics-ready temp CSV
7. upload_to_qualtrics(...)        â†’ POST + poll until complete
```

On success: popup showing OpCo + number of records submitted
On failure: popup with error message

In [21]:
# ============================================================
# EXECUTE — Full ETL Pipeline for all OpCos
# ============================================================

for opco in list(SFTP_FOLDERS.keys()):
    try:
        df = dataframes.get(opco)
        if df is None:
            print(f"[{opco}] Skipping — no data extracted")
            continue

        # Step 1 — Transform: rename columns, standardize types
        df = transform(df, opco)

        # Step 2 — Validate & tag
        df = validate_and_tag(df, opco)

        # Step 3 — Filter: completed responses only
        df_completed = filter_completed(df, opco)

        # Step 4 — Save cleaned file to SharePoint
        clean_filename = raw_files[opco][0].replace(".csv", "_cleaned.csv")
        clean_path, _  = get_output_path(clean_filename, opco)
        load_data(df_completed, clean_path)
        print(f"[{opco}] Saved cleaned file  : {clean_path}")

        # Step 5 — Prepare Qualtrics-ready format
        df_q = prepare_for_qualtrics(df_completed, opco)

        # Step 6 — Save Qualtrics-ready CSV with "Completed" suffix and upload
        original_name    = raw_files[opco][0]                              # e.g. CMP Daily Survey Report_20260615.csv
        completed_name   = original_name.replace(".csv", " Completed.csv") # e.g. CMP Daily Survey Report_20260615 Completed.csv
        completed_path, _ = get_output_path(completed_name, opco)
        load_data(df_q, completed_path)
        print(f"[{opco}] Saved Qualtrics file: {completed_path}")

        result = upload_to_qualtrics(completed_path, DATA_CENTER, SURVEYS_ID[opco], API_TOKEN)

        popup_info(
            f"Upload complete\nOpCo   : {opco}\nRecords: {len(df_completed)}\nFile   : {completed_name}",
            title=f"{opco} Upload Successful"
        )

    except Exception as e:
        error_msg = f"[{opco}] Pipeline failed:\n{e}"
        print(error_msg)
        popup_error(error_msg, title=f"{opco} ETL Error")

print("\nAll OpCos processed.")

[CMP] Transform complete | Shape: (66, 15)
[CMP] Validate & tag complete | Shape: (66, 16)
[CMP] Original records  : 66
[CMP] Completed records : 18
[CMP] Excluded          : 48
[CMP] Saved cleaned file  : C:\Users\E978423\OneDrive - IBERDROLA S.A\iQor-Avangrid - General\iQor_CMP\CMP Daily Survey Report_20260629_cleaned.csv
[CMP] Qualtrics prep complete | Shape: (20, 16)


,ID,Name,Date/Time,Work Group,Interaction ID,Phone Number,Survey Name,CSAT,NPS,I_C,C_K,FCR,Call_Reason,Survey Status Count,Survey Completion,Tag
0,ID,Name,Date/Time,Work Group,Interaction ID,Phone Number,Survey Name,CSAT,NPS,I_C,C_K,FCR,Call_Reason,Survey Status Count,Survey Completion,Tag
1,"{""ImportId"": ""QID1_TEXT""}","{""ImportId"": ""QID2_TEXT""}","{""ImportId"": ""QID3_TEXT""}","{""ImportId"": ""QID4_TEXT""}","{""ImportId"": ""QID5_TEXT""}","{""ImportId"": ""QID6_TEXT""}","{""ImportId"": ""QID7_TEXT""}","{""ImportId"": ""QID9_TEXT""}","{""ImportId"": ""QID8_TEXT""}","{""ImportId"": ""QID10_TEXT""}","{""ImportId"": ""QID11_TEXT""}","{""ImportId"": ""QID12_TEXT""}","{""ImportId"": ""QID13_TEXT""}","{""ImportId"": ""QID15_TEXT""}","{""ImportId"": ""QID16_TEXT""}","{""ImportId"": ""QID14_TEXT""}"
2,56441398,sean.case,06/29/2026 07:42:01,CMP.USUT.CS.RESCRCL,710269613439,2627252384,CMP IQR Survey w/ NPS,5,1,#,#,#,1,6,COMPLETED,
3,34795386,evonta.garrett,06/29/2026 08:46:34,CMP.USUT.CS.RESCRCL,710269644761,2077981740,CMP IQR Survey w/ NPS,5,4,#,#,#,3,6,COMPLETED,
4,39761737,brianna.gay,06/29/2026 08:55:46,CMP.USUT.CS.RESCRCL,710269657735,2073515481,CMP IQR Survey w/ NPS,5,3,#,#,#,1,6,COMPLETED,


[CMP] Saved Qualtrics file: C:\Users\E978423\OneDrive - IBERDROLA S.A\iQor-Avangrid - General\iQor_CMP\CMP Daily Survey Report_20260629 Completed.csv

Uploading: C:\Users\E978423\OneDrive - IBERDROLA S.A\iQor-Avangrid - General\iQor_CMP\CMP Daily Survey Report_20260629 Completed.csv
Job accepted | progressId: e3ff85cd-6348-4fd5-839a-ee1e076c104b
  [01] inProgress | 0%
  [02] complete | 100%
Upload confirmed complete.
[RGE] Transform complete | Shape: (198, 14)
[RGE] Validate & tag complete | Shape: (198, 15)
[RGE] Original records  : 198
[RGE] Completed records : 110
[RGE] Excluded          : 88
[RGE] Saved cleaned file  : C:\Users\E978423\OneDrive - IBERDROLA S.A\iQor-Avangrid - General\iQor_RGE\RGE Daily Survey Report_20260629_cleaned.csv
[RGE] Qualtrics prep complete | Shape: (112, 15)


,ID,Name,Date/Time,Work Group,Interaction ID,Phone Number,Survey Name,NPS,FCR,CSAT,E_H,C_E,Call Reason,Survey Status,Tag
0,ID,Name,Date/Time,Work Group,Interaction ID,Phone Number,Survey Name,NPS,FCR,CSAT,Ease of Help,Clear Explanation,Call Reason,Survey Status,Tag
1,"{""ImportId"": ""QID1_TEXT""}","{""ImportId"": ""QID2_TEXT""}","{""ImportId"": ""QID3_TEXT""}","{""ImportId"": ""QID4_TEXT""}","{""ImportId"": ""QID5_TEXT""}","{""ImportId"": ""QID6_TEXT""}","{""ImportId"": ""QID7_TEXT""}","{""ImportId"": ""QID9_TEXT""}","{""ImportId"": ""QID12_TEXT""}","{""ImportId"": ""QID8_TEXT""}","{""ImportId"": ""QID10_TEXT""}","{""ImportId"": ""QID11_TEXT""}","{""ImportId"": ""QID13_TEXT""}","{""ImportId"": ""QID14_TEXT""}","{""ImportId"": ""QID15_TEXT""}"
2,43978728,shuhun.baloch,06/29/2026 07:20:22,RGE.USUT.FE.RGCRCL,710269605057,5854450960,RGE IQR Survey w/ NPS,10,1,5,4,4,3,COMPLETED,
3,44081434,kayla.hunt1,06/29/2026 07:20:46,RGE.USUT.FE.RGCRCL,710269604699,5856988135,RGE IQR Survey w/ NPS,10,1,5,5,5,3,COMPLETED,
4,69022815,matheresita.inciong,06/29/2026 07:25:43,RGE.USUT.FE.RGCRCL,710269608000,5857647942,RGE IQR Survey w/ NPS,0,0,4,3,5,1,COMPLETED,


[RGE] Saved Qualtrics file: C:\Users\E978423\OneDrive - IBERDROLA S.A\iQor-Avangrid - General\iQor_RGE\RGE Daily Survey Report_20260629 Completed.csv

Uploading: C:\Users\E978423\OneDrive - IBERDROLA S.A\iQor-Avangrid - General\iQor_RGE\RGE Daily Survey Report_20260629 Completed.csv
Job accepted | progressId: 14190469-ce1c-4d10-b260-2316c5b323f7
  [01] inProgress | 0%
  [02] inProgress | 0%
  [03] complete | 100%
Upload confirmed complete.
[NSE] Transform complete | Shape: (391, 14)
[NSE] Validate & tag complete | Shape: (391, 15)
[NSE] Original records  : 391
[NSE] Completed records : 226
[NSE] Excluded          : 165
[NSE] Saved cleaned file  : C:\Users\E978423\OneDrive - IBERDROLA S.A\iQor-Avangrid - General\iQor_NYSEG\NSE Daily Survey Report_20260629_cleaned.csv
[NSE] Qualtrics prep complete | Shape: (228, 15)


,ID,Name,Date/Time,Work Group,Interaction ID,Phone Number,Survey Name,NPS,FCR,CSAT,E_H,C_E,Call Reason,Survey Status,Tag
0,ID,Name,Date/Time,Work Group,Interaction ID,Phone Number,Survey Name,NPS,FCR,CSAT,Ease of Help,Clear Explanation,Call Reason,Survey Status,Tag
1,"{""ImportId"": ""QID1_TEXT""}","{""ImportId"": ""QID2_TEXT""}","{""ImportId"": ""QID3_TEXT""}","{""ImportId"": ""QID4_TEXT""}","{""ImportId"": ""QID5_TEXT""}","{""ImportId"": ""QID6_TEXT""}","{""ImportId"": ""QID7_TEXT""}","{""ImportId"": ""QID9_TEXT""}","{""ImportId"": ""QID12_TEXT""}","{""ImportId"": ""QID8_TEXT""}","{""ImportId"": ""QID10_TEXT""}","{""ImportId"": ""QID11_TEXT""}","{""ImportId"": ""QID13_TEXT""}","{""ImportId"": ""QID14_TEXT""}","{""ImportId"": ""QID15_TEXT""}"
2,60920556,alexandria.colvin,06/29/2026 07:13:00,NSE.USUT.FE.GEN,710269605646,7164335011,NSE IQR Survey w/ NPS,10,1,5,1,5,3,COMPLETED,
3,69353654,symba.sebastian,06/29/2026 07:25:34,NSE.USUT.FE.GEN,710269607364,5855573534,NSE IQR Survey w/ NPS,10,1,5,5,5,1,COMPLETED,
4,38869988,sherry.stdennis,06/29/2026 07:26:24,NSE.USUT.FE.NYCRCL,710269609063,8454282579,NSE IQR Survey w/ NPS,10,1,5,5,5,3,COMPLETED,


[NSE] Saved Qualtrics file: C:\Users\E978423\OneDrive - IBERDROLA S.A\iQor-Avangrid - General\iQor_NYSEG\NSE Daily Survey Report_20260629 Completed.csv

Uploading: C:\Users\E978423\OneDrive - IBERDROLA S.A\iQor-Avangrid - General\iQor_NYSEG\NSE Daily Survey Report_20260629 Completed.csv
Job accepted | progressId: 4fe92df9-fb1b-4f83-9f38-bde1e6a52da1
  [01] inProgress | 0%
  [02] complete | 100%
Upload confirmed complete.

All OpCos processed.
